# **import stable xl and required support**

In [ ]:
from huggingface_hub import login
login("enter your api-key")

In [ ]:
!pip install diffusers transformers accelerate DeepCache --quiet

# **Observation**

In [ ]:
# ============================================================
# Delta Observation: Empirical Motivation for Non-Uniform Scheduler
# ============================================================

import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from diffusers import StableDiffusionXLPipeline, DDIMScheduler
import pandas as pd

# ── Config (edit these) ──
CSV_PATH = "prompts_LITTLE1.csv"
PROMPT_COLUMN = "prompt"
NUM_STEPS = 50
SEED = 42
NUM_OBSERVE = 50
SAVE_DIR = './delta_obs'
MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

os.makedirs(SAVE_DIR, exist_ok=True)

# ── Read CSV ──
df = pd.read_csv(CSV_PATH)
prompts = df[PROMPT_COLUMN].dropna().tolist()
print(f"Loaded {len(prompts)} prompts from '{CSV_PATH}'")

# ── Load model ──
print(f"\nLoading {MODEL_ID}...")
pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE, use_safetensors=True
)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(DEVICE)
print("Model loaded.\n")


# ── Utilities ──
def set_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def observe_ddim_delta(pipe, prompt, steps, seed):
    """Run pure DDIM, record UNet output per step, compute deltas."""
    set_seed(seed)
    outputs = []

    original_forward = pipe.unet.forward

    def hooked_forward(*args, **kwargs):
        result = original_forward(*args, **kwargs)
        out = result.sample if hasattr(result, 'sample') else result
        if isinstance(out, tuple):
            out = out[0]
        outputs.append(out.detach().clone())
        return result

    pipe.unet.forward = hooked_forward
    try:
        with torch.inference_mode():
            pipe(prompt, num_inference_steps=steps, output_type="pil").images[0]
    finally:
        pipe.unet.forward = original_forward

    step_delta = [0.0]
    cumul_delta = [0.0]
    first = outputs[0].float()
    first_norm = torch.norm(first) + 1e-8

    for t in range(1, len(outputs)):
        cur = outputs[t].float()
        prev = outputs[t - 1].float()
        step_delta.append((torch.norm(cur - prev) / (torch.norm(prev) + 1e-8)).item())
        cumul_delta.append((torch.norm(cur - first) / first_norm).item())

    return step_delta, cumul_delta


# ── Run observation ──
num_obs = min(NUM_OBSERVE, len(prompts))
all_step_deltas = []
all_cumul_deltas = []

print(f"Observing UNet output on {num_obs} prompts (pure DDIM, no cache)...\n")

for i in range(num_obs):
    p = prompts[i]
    print(f"  [{i+1}/{num_obs}] {p[:60]}{'...' if len(p) > 60 else ''}")
    sd, cd = observe_ddim_delta(pipe, p, steps=NUM_STEPS, seed=SEED)
    all_step_deltas.append(sd)
    all_cumul_deltas.append(cd)
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

# ── Aggregate ──
step_arr = np.array(all_step_deltas)
cumul_arr = np.array(all_cumul_deltas)
mean_step = np.mean(step_arr, axis=0)
std_step = np.std(step_arr, axis=0)
mean_cumul = np.mean(cumul_arr, axis=0)
std_cumul = np.std(cumul_arr, axis=0)

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ts = np.arange(NUM_STEPS)
half = NUM_STEPS // 2

ax = axes[0]
ax.plot(ts[1:], mean_step[1:], 'b-', linewidth=2, label='Mean step δ')
ax.fill_between(ts[1:], mean_step[1:] - std_step[1:], mean_step[1:] + std_step[1:],
                alpha=0.2, color='blue', label='±1 std')
first_half = np.mean(mean_step[1:half])
second_half = np.mean(mean_step[half:])
ax.axhline(first_half, xmin=0, xmax=0.5, color='red', linestyle='--',
           alpha=0.7, label=f'1st half mean={first_half:.4f}')
ax.axhline(second_half, xmin=0.5, xmax=1.0, color='green', linestyle='--',
           alpha=0.7, label=f'2nd half mean={second_half:.4f}')
ax.set_xlabel('Timestep')
ax.set_ylabel('Relative Change')
ax.set_title('Step Delta: ‖outₜ − outₜ₋₁‖ / ‖outₜ₋₁‖')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(ts, mean_cumul, 'r-', linewidth=2, label='Mean cumulative δ')
ax.fill_between(ts, mean_cumul - std_cumul, mean_cumul + std_cumul,
                alpha=0.2, color='red', label='±1 std')
ax.set_xlabel('Timestep')
ax.set_ylabel('Cumulative Change')
ax.set_title('Cumulative Delta: ‖outₜ − out₀‖ / ‖out₀‖')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle(f'UNet Output Evolution (Pure DDIM, {num_obs} prompts avg)', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'delta_observation.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── Summary ──
ratio = first_half / second_half if second_half > 0 else float('inf')
print(f"\n{'=' * 60}")
print(f"  Delta Observation Summary ({num_obs} prompts)")
print(f"{'=' * 60}")
print(f"  1st half (step 1-{half-1}) mean δ = {first_half:.4f}")
print(f"  2nd half (step {half}-{NUM_STEPS-1}) mean δ = {second_half:.4f}")
print(f"  Ratio = {ratio:.2f}x")
print()
if ratio > 1.5:
    print(f"  >> Strong support for front-dense scheduling.")
    print(f"     Early steps show {ratio:.1f}x more variation.")
elif ratio > 1.1:
    print(f"  >> Moderate support for front-dense scheduling.")
else:
    print(f"  >> Weak/no support. Uniform scheduling may suffice.")
print(f"{'=' * 60}")

# ── Clean up ──
del pipe
if DEVICE == "cuda":
    torch.cuda.empty_cache()
print("\nModel released. Ready for main experiments.")

# **Base on observation**

In [ ]:
# ============================================================
# Adaptive DeepCache (Budget-Constrained Fair Comparison)
# DDIM Baseline vs DeepCache (official) vs Adaptive Interval (δ-driven + Budget)
# ============================================================

import os
import csv
import time
import warnings
from contextlib import contextmanager
from dataclasses import dataclass
from typing import Optional, List, Set

import numpy as np
import torch
from PIL import Image, ImageDraw

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
print(f"Device: {DEVICE}, torch: {torch.__version__}")

# ============================================================
# 1. Load the diffusion model
# ============================================================

from diffusers import StableDiffusionXLPipeline, DDIMScheduler

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=DTYPE,
    variant="fp16",
    use_safetensors=True,
).to(DEVICE)

pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

# ============================================================
# 2. Evaluation metrics: PSNR, Image Similarity, CLIP Score
# ============================================================

from transformers import CLIPProcessor, CLIPModel

print("Loading CLIP...")
clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(DEVICE).eval()
clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)
print("CLIP loaded!")


def compute_psnr(img_ref, img_gen):
    a = np.array(img_ref.convert("RGB")).astype(np.float32)
    b = np.array(img_gen.convert("RGB")).astype(np.float32)
    mse = np.mean((a - b) ** 2)
    if mse == 0:
        return 100.0
    return float(20 * np.log10(255.0 / (np.sqrt(mse) + 1e-8)))


@torch.no_grad()
def compute_clip_image_similarity(img_ref, img_gen):
    inp_ref = clip_processor(images=img_ref.convert("RGB"), return_tensors="pt").to(DEVICE)
    inp_gen = clip_processor(images=img_gen.convert("RGB"), return_tensors="pt").to(DEVICE)
    f_ref = clip_model.get_image_features(**inp_ref)
    f_gen = clip_model.get_image_features(**inp_gen)
    if not isinstance(f_ref, torch.Tensor):
        f_ref = f_ref.last_hidden_state[:, 0]
    if not isinstance(f_gen, torch.Tensor):
        f_gen = f_gen.last_hidden_state[:, 0]
    f_ref = f_ref / f_ref.norm(dim=-1, keepdim=True)
    f_gen = f_gen / f_gen.norm(dim=-1, keepdim=True)
    return float((f_ref @ f_gen.T).item())


@torch.no_grad()
def compute_clip_score(image, text):
    inputs = clip_processor(text=[text], images=image.convert("RGB"),
                            return_tensors="pt", padding=True,
                            truncation=True, max_length=77)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    out = clip_model(**inputs)
    ie = out.image_embeds
    te = out.text_embeds
    if ie is None:
        ie = out.vision_model_output.last_hidden_state[:, 0]
        ie = clip_model.visual_projection(ie)
    if te is None:
        te = out.text_model_output.last_hidden_state[:, 0]
        te = clip_model.text_projection(te)
    ie = ie / ie.norm(dim=-1, keepdim=True)
    te = te / te.norm(dim=-1, keepdim=True)
    return float((ie @ te.T).item())


# ============================================================
# 3. Adaptive Interval DeepCache (δ-driven + Budget constraint)
# ============================================================

@dataclass
class AdaptiveIntervalConfig:

    interval_at_low_delta: int = 2    # Small δ (large change) → long interval
    interval_at_high_delta: int = 5   # Large δ (small change) → short interval
    delta_min: float = 0.005          # δ below this → use interval_at_low_delta
    delta_max: float = 0.05           # δ above this → use interval_at_high_delta
    cache_branch_id: int = 0
    warmup_steps: int = 2
    # Budget constraint
    total_steps: int = 50
    full_budget: int = -1


class AdaptiveIntervalHelper:

    def __init__(self, pipe, config: AdaptiveIntervalConfig):
        self.pipe = pipe
        self.config = config
        self.function_dict = {}
        self.cached_output = {}
        self.step_log = []
        self._ema_delta = 0.0

        # Step tracking
        self.steps_done = 0
        self.steps_since_full = 0           # Steps since last full computation
        self.current_interval = config.interval_at_low_delta  # Current adaptive interval
        self.last_output_tensor = None      # Previous UNet output for δ computation
        self.last_delta = 0.0               # Most recent δ value
        self._is_full_step = True           # Flag read by block-level wrappers

        # Budget tracking
        self._budget_enabled = (config.full_budget > 0)
        self._remaining_budget = config.full_budget if self._budget_enabled else float('inf')
        self._total_steps = config.total_steps

        # Spatial cache boundary (same mapping as official DeepCache)
        self.cache_block_id = config.cache_branch_id // 3
        self.cache_layer_id = config.cache_branch_id % 3

    def _compute_interval(self, delta):

        cfg = self.config
        if delta <= cfg.delta_min:
            return cfg.interval_at_low_delta
        if delta >= cfg.delta_max:
            return cfg.interval_at_high_delta
        ratio = (delta - cfg.delta_min) / (cfg.delta_max - cfg.delta_min)
        interval = cfg.interval_at_low_delta + ratio * (cfg.interval_at_high_delta - cfg.interval_at_low_delta)
        return max(1, round(interval))

    def _should_full(self):

        step = self.steps_done
        remaining_steps = self._total_steps - step  # Steps left including current
        budget = self._remaining_budget

        if self._budget_enabled:
            # Rule 1: budget exhausted → force cache
            if budget <= 0:
                return False

            # Rule 2: remaining steps == remaining budget → every remaining step must be full
            # This guarantees all budget is consumed (no waste)
            if remaining_steps <= budget:
                return True

        # Rule 3: warmup period → full (need initial cache to be established)
        if step < self.config.warmup_steps:
            return True

        # Rule 4: δ-adaptive interval reached → time for a full step
        if self.steps_since_full >= self.current_interval:
            return True

        # Rule 5: budget backfill safeguard
        # Prevents the adaptive logic from being too conservative early on,
        # which would force all remaining budget into the final steps
        if self._budget_enabled:
            max_cache_steps = remaining_steps - budget  # Max cache steps we can still afford
            # Count how many consecutive cache steps we've had
            consecutive_cache = 0
            for log in reversed(self.step_log):
                if log['is_full']:
                    break
                consecutive_cache += 1
            # If we've cached as much as we can afford, force a full step now
            if consecutive_cache >= max_cache_steps:
                return True

        return False

    def _is_skip_block(self, block_i, layer_i, blocktype="down"):

        if block_i > self.cache_block_id or blocktype == 'mid':
            return True
        if block_i < self.cache_block_id:
            return False
        if blocktype == 'down':
            return layer_i >= self.cache_layer_id
        else:
            return layer_i > self.cache_layer_id

    def _wrap_unet_forward(self):

        self.function_dict['unet_forward'] = self.pipe.unet.forward
        helper = self

        def wrapped_forward(*args, **kwargs):
            # Decide full or cache via unified scheduling logic
            is_full = helper._should_full()
            helper._is_full_step = is_full

            # Execute UNet forward (block-level wrappers handle spatial caching)
            result = helper.function_dict['unet_forward'](*args, **kwargs)

            # Extract output tensor for δ computation
            out = result
            if hasattr(out, 'sample'):
                out = out.sample
            if isinstance(out, tuple):
                out = out[0]
            if isinstance(out, tuple):
                out = out[0]

            # On full steps: compute δ and update adaptive interval
            # δ = ‖output_t - output_{t-1}‖ / ‖output_{t-1}‖
            if is_full:
                with torch.no_grad():
                    if helper.last_output_tensor is not None:
                        diff = torch.norm(out.float() - helper.last_output_tensor.float())
                        norm = torch.norm(helper.last_output_tensor.float()) + 1e-8
                        raw_delta = (diff / norm).item() / max(helper.steps_since_full, 1)
                        helper._ema_delta = 0.3 * raw_delta + 0.7 * helper._ema_delta
                        helper.last_delta = helper._ema_delta
                    else:
                        helper.last_delta = 0.0
                    helper.last_output_tensor = out.detach().clone()

                # Update interval based on new δ for next scheduling decision
                helper.current_interval = helper._compute_interval(helper.last_delta)
                helper.steps_since_full = 0
                helper._remaining_budget -= 1
            else:
                helper.steps_since_full += 1

            # Log this step for analysis
            helper.step_log.append({
                'step': helper.steps_done,
                'is_full': is_full,
                'delta': helper.last_delta,
                'interval': helper.current_interval,
                'remaining_budget': helper._remaining_budget if helper._budget_enabled else -1,
            })
            helper.steps_done += 1
            return result

        self.pipe.unet.forward = wrapped_forward

    def _wrap_block_forward(self, block, block_name, block_i, layer_i, blocktype="down"):

        key = (blocktype, block_name, block_i, layer_i)
        self.function_dict[key] = block.forward
        helper = self

        def wrapped_forward(*args, **kwargs):
            if helper._is_full_step:
                # Full step: compute and cache
                result = helper.function_dict[key](*args, **kwargs)
                helper.cached_output[key] = result
                return result
            else:
                # Cache step: skip deep modules, recompute shallow ones
                if helper._is_skip_block(block_i, layer_i, blocktype) and key in helper.cached_output:
                    return helper.cached_output[key]
                result = helper.function_dict[key](*args, **kwargs)
                helper.cached_output[key] = result
                return result

        block.forward = wrapped_forward

    def _wrap_modules(self):

        self._wrap_unet_forward()

        # Down blocks: natural order
        for bi, blk in enumerate(self.pipe.unet.down_blocks):
            for li, a in enumerate(getattr(blk, "attentions", [])):
                self._wrap_block_forward(a, "attentions", bi, li, "down")
            for li, r in enumerate(getattr(blk, "resnets", [])):
                self._wrap_block_forward(r, "resnet", bi, li, "down")
            for d in (getattr(blk, "downsamplers", []) if blk.downsamplers else []):
                self._wrap_block_forward(d, "downsampler", bi, len(getattr(blk, "resnets", [])), "down")

        # Mid block
        self._wrap_block_forward(self.pipe.unet.mid_block, "mid_block", 0, 0, "mid")

        # Up blocks: both block and layer indices reversed
        bn = len(self.pipe.unet.up_blocks)
        for bi, blk in enumerate(self.pipe.unet.up_blocks):
            mapped_bi = bn - bi - 1                            # Block index reversed
            ln = len(getattr(blk, "resnets", []))              # ← ADDED
            for li, a in enumerate(getattr(blk, "attentions", [])):
                self._wrap_block_forward(a, "attentions", mapped_bi, ln-li-1, "up")   # ← FIXED
            for li, r in enumerate(getattr(blk, "resnets", [])):
                self._wrap_block_forward(r, "resnet", mapped_bi, ln-li-1, "up")       # ← FIXED
            for u in (getattr(blk, "upsamplers", []) if blk.upsamplers else []):
                self._wrap_block_forward(u, "upsampler", mapped_bi, 0, "up")

    def _unwrap_modules(self):
        """Restore all original forward functions. Mirrors _wrap_modules exactly."""
        self.pipe.unet.forward = self.function_dict['unet_forward']

        # Down blocks
        for bi, blk in enumerate(self.pipe.unet.down_blocks):
            for li, a in enumerate(getattr(blk, "attentions", [])):
                a.forward = self.function_dict[("down", "attentions", bi, li)]
            for li, r in enumerate(getattr(blk, "resnets", [])):
                r.forward = self.function_dict[("down", "resnet", bi, li)]
            for d in (getattr(blk, "downsamplers", []) if blk.downsamplers else []):
                d.forward = self.function_dict[("down", "downsampler", bi, len(getattr(blk, "resnets", [])))]

        # Mid block
        self.pipe.unet.mid_block.forward = self.function_dict[("mid", "mid_block", 0, 0)]

        # Up blocks: same reversed indexing
        bn = len(self.pipe.unet.up_blocks)
        for bi, blk in enumerate(self.pipe.unet.up_blocks):
            mapped_bi = bn - bi - 1
            ln = len(getattr(blk, "resnets", []))              # ← ADDED
            for li, a in enumerate(getattr(blk, "attentions", [])):
                a.forward = self.function_dict[("up", "attentions", mapped_bi, ln-li-1)]   # ← FIXED
            for li, r in enumerate(getattr(blk, "resnets", [])):
                r.forward = self.function_dict[("up", "resnet", mapped_bi, ln-li-1)]       # ← FIXED
            for u in (getattr(blk, "upsamplers", []) if blk.upsamplers else []):
                u.forward = self.function_dict[("up", "upsampler", mapped_bi, 0)]

    def enable(self):
        self._reset()
        self._wrap_modules()

    def disable(self):
        self._unwrap_modules()
        self._reset()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def _reset(self):
        self.function_dict = {}
        self.cached_output = {}
        self.step_log = []
        self._ema_delta = 0.0
        self.steps_done = 0
        self.steps_since_full = 0
        self.current_interval = self.config.interval_at_low_delta
        self.last_output_tensor = None
        self.last_delta = 0.0
        self._is_full_step = True
        self._remaining_budget = self.config.full_budget if self._budget_enabled else float('inf')

    @contextmanager
    def enabled(self):
        self.enable()
        try:
            yield self
        finally:
            self.disable()

    def get_stats(self):
        if not self.step_log:
            return {}
        full_steps = sum(1 for s in self.step_log if s['is_full'])
        cache_steps = sum(1 for s in self.step_log if not s['is_full'])
        return {
            'total_steps': len(self.step_log),
            'full_steps': full_steps,
            'cache_steps': cache_steps,
            'delta_trace': [s['delta'] for s in self.step_log],
            'interval_trace': [s['interval'] for s in self.step_log],
            'full_trace': ['F' if s['is_full'] else 's' for s in self.step_log],
            'budget_trace': [s['remaining_budget'] for s in self.step_log],
        }


# ============================================================
# 4. Budget computation utility
# ============================================================

def compute_deepcache_budget(total_steps: int, cache_interval: int) -> int:
    K = total_steps // cache_interval + (1 if total_steps % cache_interval else 0)
    return K


# ============================================================
# 5. Generation functions
# ============================================================

def set_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)


def gen_baseline(prompt, steps=50, seed=42):
    set_seed(seed)
    t0 = time.time()
    with torch.inference_mode():
        img = pipe(prompt, num_inference_steps=steps, output_type="pil").images[0]
    return img, time.time() - t0


def gen_deepcache(prompt, steps=50, seed=42, cache_interval=3, cache_branch_id=0):
    from DeepCache import DeepCacheSDHelper
    set_seed(seed)
    h = DeepCacheSDHelper(pipe=pipe)
    h.set_params(cache_interval=cache_interval, cache_branch_id=cache_branch_id)
    h.enable()
    t0 = time.time()
    with torch.inference_mode():
        img = pipe(prompt, num_inference_steps=steps, output_type="pil").images[0]
    elapsed = time.time() - t0
    h.disable()
    return img, elapsed


def gen_adaptive(prompt, steps=50, seed=42, config=None):
    set_seed(seed)
    if config is None:
        config = AdaptiveIntervalConfig()
    h = AdaptiveIntervalHelper(pipe, config)
    with h.enabled():
        t0 = time.time()
        with torch.inference_mode():
            img = pipe(prompt, num_inference_steps=steps, output_type="pil").images[0]
        elapsed = time.time() - t0
        stats = h.get_stats()
    return img, elapsed, stats


# ============================================================
# 6. Run experiment
# ============================================================

# ── Configuration ──
CSV_PATH = "prompts_LITTLE1.csv"
PROMPT_COLUMN = "prompt"
NUM_STEPS = 50
SEED = 42
DC_INTERVAL = 5
DC_BRANCH = 3
SAVE_DIR = "results"

# ── Compute full UNet budget from DeepCache settings ──
FULL_BUDGET = compute_deepcache_budget(NUM_STEPS, DC_INTERVAL)
print(f"\nDeepCache config: interval={DC_INTERVAL}, branch={DC_BRANCH}")
print(f"Computed Full Budget K = {FULL_BUDGET}  (out of {NUM_STEPS} total steps)")
print(f"This means: {FULL_BUDGET} Full UNet + {NUM_STEPS - FULL_BUDGET} Cached steps\n")

# ── Build adaptive config with budget constraint ──
adaptive_config = AdaptiveIntervalConfig(
    interval_at_low_delta=7,
    interval_at_high_delta=2,
    delta_min=0.002,
    delta_max=0.015,
    cache_branch_id=DC_BRANCH,
    warmup_steps=2,
    total_steps=NUM_STEPS,
    full_budget=FULL_BUDGET,
)

# ── Load prompts ──
prompts = []
with open(CSV_PATH, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        t = row.get(PROMPT_COLUMN, "").strip()
        if t:
            prompts.append(t)
print(f"{len(prompts)} prompts loaded from {CSV_PATH}\n")

# ── Run all prompts ──
os.makedirs(SAVE_DIR, exist_ok=True)
all_results = []

out_csv = os.path.join(SAVE_DIR, "results.csv")
out_fields = ["run", "prompt", "time_bl", "time_dc", "time_adc",
              "speedup_dc", "speedup_adc", "psnr_dc", "psnr_adc",
              "img_sim_dc", "img_sim_adc", "clip_bl", "clip_dc", "clip_adc",
              "dc_full", "dc_cache", "adc_full", "adc_cache",
              "budget_K", "budget_match",
              "adc_intervals", "adc_trace"]

with open(out_csv, "w", newline="", encoding="utf-8") as cf:
    cw = csv.DictWriter(cf, fieldnames=out_fields)
    cw.writeheader()

    for i, prompt in enumerate(prompts):
        print(f"\n{'='*60}\n  Run {i+1}/{len(prompts)}  [Budget K={FULL_BUDGET}]\n{'='*60}")
        print(f'  Prompt: "{prompt[:80]}{"..." if len(prompt)>80 else ""}"')

        # Step 1: Baseline (full DDIM, no caching)
        print("  [1/3] Baseline ...")
        img_bl, t_bl = gen_baseline(prompt, NUM_STEPS, SEED)
        print(f"        {t_bl:.2f}s")

        # Step 2: Official DeepCache (uniform caching)
        print(f"  [2/3] DeepCache (interval={DC_INTERVAL}, branch={DC_BRANCH}) ...")
        img_dc, t_dc = gen_deepcache(prompt, NUM_STEPS, SEED, DC_INTERVAL, DC_BRANCH)
        sp_dc = t_bl / max(t_dc, 1e-6)
        print(f"        {t_dc:.2f}s  ({sp_dc:.2f}x)")

        # Step 3: Adaptive interval with budget constraint
        print(f"  [3/3] Adaptive Interval (budget K={FULL_BUDGET}) ...")
        img_ad, t_ad, stats = gen_adaptive(prompt, NUM_STEPS, SEED, adaptive_config)
        sp_ad = t_bl / max(t_ad, 1e-6)
        ft = ''.join(stats.get('full_trace', []))
        adc_full = stats.get('full_steps', 0)
        adc_cache = stats.get('cache_steps', 0)
        budget_match = (adc_full == FULL_BUDGET)

        print(f"        {t_ad:.2f}s  ({sp_ad:.2f}x)")
        print(f"        Full={adc_full} Cache={adc_cache}  Budget match={'MATCH' if budget_match else 'MISMATCH!'}")
        print(f"        Trace: {ft}")
        print(f"        Intervals: {stats.get('interval_trace', [])}")
        print(f"        Deltas: {[f'{d:.4f}' for d in stats.get('delta_trace', [])]}")

        if not budget_match:
            print(f"  WARNING: Adaptive used {adc_full} Full but budget is {FULL_BUDGET}!")

        # DeepCache full/cache step counts (for verification)
        dc_full = FULL_BUDGET
        dc_cache = NUM_STEPS - dc_full

        # Compute all quality metrics
        psnr_dc = compute_psnr(img_bl, img_dc)
        psnr_ad = compute_psnr(img_bl, img_ad)
        sim_dc = compute_clip_image_similarity(img_bl, img_dc)
        sim_ad = compute_clip_image_similarity(img_bl, img_ad)
        clip_bl = compute_clip_score(img_bl, prompt)
        clip_dc = compute_clip_score(img_dc, prompt)
        clip_ad = compute_clip_score(img_ad, prompt)

        print(f"  PSNR    DC={psnr_dc:.1f}dB  ADC={psnr_ad:.1f}dB")
        print(f"  ImgSim  DC={sim_dc:.4f}  ADC={sim_ad:.4f}")
        print(f"  CLIP    BL={clip_bl:.4f}  DC={clip_dc:.4f}  ADC={clip_ad:.4f}")

        # Save side-by-side comparison image
        W, H = img_bl.size
        comp = Image.new("RGB", (W * 3 + 20, H + 30), "white")
        draw = ImageDraw.Draw(comp)
        for j, (img, title) in enumerate(zip(
            [img_bl, img_dc, img_ad],
            ["Baseline",
             f"DeepCache(i={DC_INTERVAL},F={dc_full})",
             f"Adaptive(F={adc_full})"]
        )):
            x = j * (W + 10)
            comp.paste(img, (x, 25))
            draw.text((x + W // 2, 5), title, fill="black", anchor="mt")
        comp_path = os.path.join(SAVE_DIR, f"cmp_{i+1}_s{SEED}.png")
        comp.save(comp_path)
        print(f"  Saved -> {comp_path}")
        try:
            from IPython.display import display
            display(comp)
        except:
            pass

        # Record results
        r = {
            'prompt': prompt,
            'times': {'baseline': t_bl, 'deepcache': t_dc, 'adaptive': t_ad},
            'speedups': {'deepcache': sp_dc, 'adaptive': sp_ad},
            'psnr': {'deepcache': psnr_dc, 'adaptive': psnr_ad},
            'img_sim': {'deepcache': sim_dc, 'adaptive': sim_ad},
            'clip': {'baseline': clip_bl, 'deepcache': clip_dc, 'adaptive': clip_ad},
            'stats': stats,
            'budget_match': budget_match,
        }
        all_results.append(r)

        # Write CSV row
        cw.writerow({
            "run": i + 1, "prompt": prompt[:100],
            "time_bl": f"{t_bl:.3f}", "time_dc": f"{t_dc:.3f}", "time_adc": f"{t_ad:.3f}",
            "speedup_dc": f"{sp_dc:.3f}", "speedup_adc": f"{sp_ad:.3f}",
            "psnr_dc": f"{psnr_dc:.2f}", "psnr_adc": f"{psnr_ad:.2f}",
            "img_sim_dc": f"{sim_dc:.4f}", "img_sim_adc": f"{sim_ad:.4f}",
            "clip_bl": f"{clip_bl:.4f}", "clip_dc": f"{clip_dc:.4f}", "clip_adc": f"{clip_ad:.4f}",
            "dc_full": dc_full, "dc_cache": dc_cache,
            "adc_full": adc_full, "adc_cache": adc_cache,
            "budget_K": FULL_BUDGET,
            "budget_match": "YES" if budget_match else "NO",
            "adc_intervals": str(stats.get('interval_trace', [])),
            "adc_trace": ft,
        })
        cf.flush()

# ============================================================
# 7. Summary
# ============================================================

print(f"\n{'='*70}")
print(f"  SUMMARY ({len(all_results)} runs, Budget K={FULL_BUDGET})")
print(f"{'='*70}")
sp_dc = np.mean([r['speedups']['deepcache'] for r in all_results])
sp_ad = np.mean([r['speedups']['adaptive'] for r in all_results])
p_dc = np.mean([r['psnr']['deepcache'] for r in all_results])
p_ad = np.mean([r['psnr']['adaptive'] for r in all_results])
s_dc = np.mean([r['img_sim']['deepcache'] for r in all_results])
s_ad = np.mean([r['img_sim']['adaptive'] for r in all_results])
c_bl = np.mean([r['clip']['baseline'] for r in all_results])
c_dc = np.mean([r['clip']['deepcache'] for r in all_results])
c_ad = np.mean([r['clip']['adaptive'] for r in all_results])
budget_ok = sum(1 for r in all_results if r['budget_match'])

print(f"  Budget match: {budget_ok}/{len(all_results)} runs")
print()
print(f"  {'Method':<28} {'Speedup':<10} {'PSNR(dB)':<12} {'ImgSim':<10} {'CLIP':<10} {'Full Steps'}")
print(f"  {'-'*82}")
print(f"  {'DDIM Baseline':<28} {'1.00x':<10} {'--':<12} {'--':<10} {c_bl:.4f}{'':>4} {NUM_STEPS}")
print(f"  {'DeepCache(i='+str(DC_INTERVAL)+')':<28} {sp_dc:.2f}x{'':>4} {p_dc:.1f}{'':>7} {s_dc:.4f}{'':>3} {c_dc:.4f}{'':>3} {FULL_BUDGET}")
print(f"  {'Adaptive(budget='+str(FULL_BUDGET)+')':<28} {sp_ad:.2f}x{'':>4} {p_ad:.1f}{'':>7} {s_ad:.4f}{'':>3} {c_ad:.4f}{'':>3} {FULL_BUDGET}")
print()
print(f"  Both methods use exactly {FULL_BUDGET} full UNet evaluations.")
print(f"  Only difference: DeepCache=uniform (every {DC_INTERVAL} steps), Adaptive=δ-driven (front-heavy).")
print(f"{'='*70}")
print(f"  CSV -> {out_csv}")
print("Done!")

# **REVERSE**

In [ ]:
# ============================================================
# Adaptive DeepCache (Budget-Constrained Fair Comparison)
# DDIM Baseline vs DeepCache (official) vs Adaptive Interval (δ-driven + Budget)
# Reverse the logic of the above
# ============================================================

import os
import csv
import time
import warnings
from contextlib import contextmanager
from dataclasses import dataclass
from typing import Optional, List, Set

import numpy as np
import torch
from PIL import Image, ImageDraw

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
print(f"Device: {DEVICE}, torch: {torch.__version__}")

# ============================================================
# 1. Load the diffusion model
# ============================================================

from diffusers import StableDiffusionXLPipeline, DDIMScheduler

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=DTYPE,
    variant="fp16",
    use_safetensors=True,
).to(DEVICE)

pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

# ============================================================
# 2. Evaluation metrics: PSNR, Image Similarity, CLIP Score
# ============================================================

from transformers import CLIPProcessor, CLIPModel

print("Loading CLIP...")
clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(DEVICE).eval()
clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)
print("CLIP loaded!")


def compute_psnr(img_ref, img_gen):
    """Compute Peak Signal-to-Noise Ratio (pixel-level fidelity, higher=better)."""
    a = np.array(img_ref.convert("RGB")).astype(np.float32)
    b = np.array(img_gen.convert("RGB")).astype(np.float32)
    mse = np.mean((a - b) ** 2)
    if mse == 0:
        return 100.0
    return float(20 * np.log10(255.0 / (np.sqrt(mse) + 1e-8)))


@torch.no_grad()
def compute_clip_image_similarity(img_ref, img_gen):
    """Compute CLIP-based cosine similarity between two images (semantic similarity)."""
    inp_ref = clip_processor(images=img_ref.convert("RGB"), return_tensors="pt").to(DEVICE)
    inp_gen = clip_processor(images=img_gen.convert("RGB"), return_tensors="pt").to(DEVICE)
    f_ref = clip_model.get_image_features(**inp_ref)
    f_gen = clip_model.get_image_features(**inp_gen)
    if not isinstance(f_ref, torch.Tensor):
        f_ref = f_ref.last_hidden_state[:, 0]
    if not isinstance(f_gen, torch.Tensor):
        f_gen = f_gen.last_hidden_state[:, 0]
    f_ref = f_ref / f_ref.norm(dim=-1, keepdim=True)
    f_gen = f_gen / f_gen.norm(dim=-1, keepdim=True)
    return float((f_ref @ f_gen.T).item())


@torch.no_grad()
def compute_clip_score(image, text):
    """Compute CLIP score measuring text-image alignment (prompt faithfulness)."""
    inputs = clip_processor(text=[text], images=image.convert("RGB"),
                            return_tensors="pt", padding=True,
                            truncation=True, max_length=77)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    out = clip_model(**inputs)
    ie = out.image_embeds
    te = out.text_embeds
    if ie is None:
        ie = out.vision_model_output.last_hidden_state[:, 0]
        ie = clip_model.visual_projection(ie)
    if te is None:
        te = out.text_model_output.last_hidden_state[:, 0]
        te = clip_model.text_projection(te)
    ie = ie / ie.norm(dim=-1, keepdim=True)
    te = te / te.norm(dim=-1, keepdim=True)
    return float((ie @ te.T).item())


# ============================================================
# 3. Adaptive Interval DeepCache (δ-driven + Budget constraint)
# ============================================================

@dataclass
class AdaptiveIntervalConfig:

    interval_at_low_delta: int = 2    # Small δ (large change) → short interval
    interval_at_high_delta: int = 5   # Large δ (small change) → long interval
    delta_min: float = 0.005          # δ below this → use interval_at_low_delta
    delta_max: float = 0.05           # δ above this → use interval_at_high_delta
    cache_branch_id: int = 0
    warmup_steps: int = 2
    # Budget constraint
    total_steps: int = 50
    full_budget: int = -1


class AdaptiveIntervalHelper:

    def __init__(self, pipe, config: AdaptiveIntervalConfig):
        self.pipe = pipe
        self.config = config
        self.function_dict = {}
        self.cached_output = {}
        self.step_log = []

        # Step tracking
        self.steps_done = 0
        self.steps_since_full = 0
        self.current_interval = config.interval_at_low_delta
        self.last_output_tensor = None
        self.last_delta = 0.0
        self._is_full_step = True

        # Budget tracking
        self._budget_enabled = (config.full_budget > 0)
        self._remaining_budget = config.full_budget if self._budget_enabled else float('inf')
        self._total_steps = config.total_steps

        # Spatial cache boundary (same mapping as official DeepCache)
        self.cache_block_id = config.cache_branch_id // 3
        self.cache_layer_id = config.cache_branch_id % 3

    def _compute_interval(self, delta):

        cfg = self.config
        if delta <= cfg.delta_min:
            return cfg.interval_at_low_delta
        if delta >= cfg.delta_max:
            return cfg.interval_at_high_delta
        ratio = (delta - cfg.delta_min) / (cfg.delta_max - cfg.delta_min)
        interval = cfg.interval_at_low_delta + ratio * (cfg.interval_at_high_delta - cfg.interval_at_low_delta)
        return max(1, round(interval))

    def _should_full(self):

        step = self.steps_done
        remaining_steps = self._total_steps - step
        budget = self._remaining_budget

        if self._budget_enabled:
            # Rule 1: budget exhausted → force cache
            if budget <= 0:
                return False

            # Rule 2: remaining steps == remaining budget → every remaining step must be full
            # This guarantees all budget is consumed (no waste)
            if remaining_steps <= budget:
                return True

        # Rule 3: warmup period → full (need initial cache to be established)
        if step < self.config.warmup_steps:
            return True

        # Rule 4: δ-adaptive interval reached → time for a full step
        if self.steps_since_full >= self.current_interval:
            return True

        # Rule 5: budget backfill safeguard
        # Prevents the adaptive logic from being too conservative early on,
        # which would force all remaining budget into the final steps
        if self._budget_enabled:
            max_cache_steps = remaining_steps - budget  # Max cache steps we can still afford
            # Count how many consecutive cache steps we've had
            consecutive_cache = 0
            for log in reversed(self.step_log):
                if log['is_full']:
                    break
                consecutive_cache += 1
            # If we've cached as much as we can afford, force a full step now
            if consecutive_cache >= max_cache_steps:
                return True

        return False

    def _is_skip_block(self, block_i, layer_i, blocktype="down"):

        if block_i > self.cache_block_id or blocktype == 'mid':
            return True
        if block_i < self.cache_block_id:
            return False
        if blocktype == 'down':
            return layer_i >= self.cache_layer_id
        else:
            return layer_i > self.cache_layer_id

    def _wrap_unet_forward(self):

        self.function_dict['unet_forward'] = self.pipe.unet.forward
        helper = self

        def wrapped_forward(*args, **kwargs):

            is_full = helper._should_full()
            helper._is_full_step = is_full

            result = helper.function_dict['unet_forward'](*args, **kwargs)

            out = result
            if hasattr(out, 'sample'):
                out = out.sample
            if isinstance(out, tuple):
                out = out[0]
            if isinstance(out, tuple):
                out = out[0]

            if is_full:
                with torch.no_grad():
                    if helper.last_output_tensor is not None:
                        diff = torch.norm(out.float() - helper.last_output_tensor.float())
                        norm = torch.norm(helper.last_output_tensor.float()) + 1e-8
                        helper.last_delta = (diff / norm).item() / max(helper.steps_since_full, 1)
                    else:
                        helper.last_delta = 0.0
                    helper.last_output_tensor = out.detach().clone()

                helper.current_interval = helper._compute_interval(helper.last_delta)
                helper.steps_since_full = 0
                helper._remaining_budget -= 1
            else:
                helper.steps_since_full += 1

            helper.step_log.append({
                'step': helper.steps_done,
                'is_full': is_full,
                'delta': helper.last_delta,
                'interval': helper.current_interval,
                'remaining_budget': helper._remaining_budget if helper._budget_enabled else -1,
            })
            helper.steps_done += 1
            return result

        self.pipe.unet.forward = wrapped_forward

    def _wrap_block_forward(self, block, block_name, block_i, layer_i, blocktype="down"):

        key = (blocktype, block_name, block_i, layer_i)
        self.function_dict[key] = block.forward
        helper = self

        def wrapped_forward(*args, **kwargs):
            if helper._is_full_step:
                # Full step: compute and cache
                result = helper.function_dict[key](*args, **kwargs)
                helper.cached_output[key] = result
                return result
            else:
                # Cache step: skip deep modules, recompute shallow ones
                if helper._is_skip_block(block_i, layer_i, blocktype) and key in helper.cached_output:
                    return helper.cached_output[key]
                result = helper.function_dict[key](*args, **kwargs)
                helper.cached_output[key] = result
                return result

        block.forward = wrapped_forward

    def _wrap_modules(self):

        self._wrap_unet_forward()

        # Down blocks: natural order
        for bi, blk in enumerate(self.pipe.unet.down_blocks):
            for li, a in enumerate(getattr(blk, "attentions", [])):
                self._wrap_block_forward(a, "attentions", bi, li, "down")
            for li, r in enumerate(getattr(blk, "resnets", [])):
                self._wrap_block_forward(r, "resnet", bi, li, "down")
            for d in (getattr(blk, "downsamplers", []) if blk.downsamplers else []):
                self._wrap_block_forward(d, "downsampler", bi, len(getattr(blk, "resnets", [])), "down")

        # Mid block
        self._wrap_block_forward(self.pipe.unet.mid_block, "mid_block", 0, 0, "mid")

        # Up blocks: both block and layer indices reversed
        bn = len(self.pipe.unet.up_blocks)
        for bi, blk in enumerate(self.pipe.unet.up_blocks):
            mapped_bi = bn - bi - 1                            # Block index reversed
            ln = len(getattr(blk, "resnets", []))              # ← ADDED
            for li, a in enumerate(getattr(blk, "attentions", [])):
                self._wrap_block_forward(a, "attentions", mapped_bi, ln-li-1, "up")   # ← FIXED
            for li, r in enumerate(getattr(blk, "resnets", [])):
                self._wrap_block_forward(r, "resnet", mapped_bi, ln-li-1, "up")       # ← FIXED
            for u in (getattr(blk, "upsamplers", []) if blk.upsamplers else []):
                self._wrap_block_forward(u, "upsampler", mapped_bi, 0, "up")

    def _unwrap_modules(self):
        """Restore all original forward functions. Mirrors _wrap_modules exactly."""
        self.pipe.unet.forward = self.function_dict['unet_forward']

        # Down blocks
        for bi, blk in enumerate(self.pipe.unet.down_blocks):
            for li, a in enumerate(getattr(blk, "attentions", [])):
                a.forward = self.function_dict[("down", "attentions", bi, li)]
            for li, r in enumerate(getattr(blk, "resnets", [])):
                r.forward = self.function_dict[("down", "resnet", bi, li)]
            for d in (getattr(blk, "downsamplers", []) if blk.downsamplers else []):
                d.forward = self.function_dict[("down", "downsampler", bi, len(getattr(blk, "resnets", [])))]

        # Mid block
        self.pipe.unet.mid_block.forward = self.function_dict[("mid", "mid_block", 0, 0)]

        # Up blocks: same reversed indexing
        bn = len(self.pipe.unet.up_blocks)
        for bi, blk in enumerate(self.pipe.unet.up_blocks):
            mapped_bi = bn - bi - 1
            ln = len(getattr(blk, "resnets", []))              # ← ADDED
            for li, a in enumerate(getattr(blk, "attentions", [])):
                a.forward = self.function_dict[("up", "attentions", mapped_bi, ln-li-1)]   # ← FIXED
            for li, r in enumerate(getattr(blk, "resnets", [])):
                r.forward = self.function_dict[("up", "resnet", mapped_bi, ln-li-1)]       # ← FIXED
            for u in (getattr(blk, "upsamplers", []) if blk.upsamplers else []):
                u.forward = self.function_dict[("up", "upsampler", mapped_bi, 0)]

    def enable(self):
        self._reset()
        self._wrap_modules()

    def disable(self):
        self._unwrap_modules()
        self._reset()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def _reset(self):
        self.function_dict = {}
        self.cached_output = {}
        self.step_log = []
        self.steps_done = 0
        self.steps_since_full = 0
        self.current_interval = self.config.interval_at_low_delta
        self.last_output_tensor = None
        self.last_delta = 0.0
        self._is_full_step = True
        self._remaining_budget = self.config.full_budget if self._budget_enabled else float('inf')

    @contextmanager
    def enabled(self):
        self.enable()
        try:
            yield self
        finally:
            self.disable()

    def get_stats(self):
        if not self.step_log:
            return {}
        full_steps = sum(1 for s in self.step_log if s['is_full'])
        cache_steps = sum(1 for s in self.step_log if not s['is_full'])
        return {
            'total_steps': len(self.step_log),
            'full_steps': full_steps,
            'cache_steps': cache_steps,
            'delta_trace': [s['delta'] for s in self.step_log],
            'interval_trace': [s['interval'] for s in self.step_log],
            'full_trace': ['F' if s['is_full'] else 's' for s in self.step_log],
            'budget_trace': [s['remaining_budget'] for s in self.step_log],
        }


# ============================================================
# 4. Budget computation utility
# ============================================================

def compute_deepcache_budget(total_steps: int, cache_interval: int) -> int:

    K = total_steps // cache_interval + (1 if total_steps % cache_interval else 0)
    return K


# ============================================================
# 5. Generation functions
# ============================================================

def set_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)


def gen_baseline(prompt, steps=50, seed=42):
    set_seed(seed)
    t0 = time.time()
    with torch.inference_mode():
        img = pipe(prompt, num_inference_steps=steps, output_type="pil").images[0]
    return img, time.time() - t0


def gen_deepcache(prompt, steps=50, seed=42, cache_interval=3, cache_branch_id=0):
    from DeepCache import DeepCacheSDHelper
    set_seed(seed)
    h = DeepCacheSDHelper(pipe=pipe)
    h.set_params(cache_interval=cache_interval, cache_branch_id=cache_branch_id)
    h.enable()
    t0 = time.time()
    with torch.inference_mode():
        img = pipe(prompt, num_inference_steps=steps, output_type="pil").images[0]
    elapsed = time.time() - t0
    h.disable()
    return img, elapsed


def gen_adaptive(prompt, steps=50, seed=42, config=None):
    set_seed(seed)
    if config is None:
        config = AdaptiveIntervalConfig()
    h = AdaptiveIntervalHelper(pipe, config)
    with h.enabled():
        t0 = time.time()
        with torch.inference_mode():
            img = pipe(prompt, num_inference_steps=steps, output_type="pil").images[0]
        elapsed = time.time() - t0
        stats = h.get_stats()
    return img, elapsed, stats


# ============================================================
# 6. Run experiment
# ============================================================

# ── Configuration ──
CSV_PATH = "prompts_LITTLE4.csv"
PROMPT_COLUMN = "prompt"
NUM_STEPS = 50
SEED = 42
DC_INTERVAL = 5
DC_BRANCH = 3
SAVE_DIR = "results"

FULL_BUDGET = compute_deepcache_budget(NUM_STEPS, DC_INTERVAL)
print(f"\nDeepCache config: interval={DC_INTERVAL}, branch={DC_BRANCH}")
print(f"Computed Full Budget K = {FULL_BUDGET}  (out of {NUM_STEPS} total steps)")
print(f"This means: {FULL_BUDGET} Full UNet + {NUM_STEPS - FULL_BUDGET} Cached steps\n")


adaptive_config = AdaptiveIntervalConfig(
    interval_at_low_delta=2,
    interval_at_high_delta=7,
    delta_min=0.005,
    delta_max=0.015,
    cache_branch_id=DC_BRANCH,
    warmup_steps=2,
    total_steps=NUM_STEPS,
    full_budget=FULL_BUDGET,
)


prompts = []
with open(CSV_PATH, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        t = row.get(PROMPT_COLUMN, "").strip()
        if t:
            prompts.append(t)
print(f"{len(prompts)} prompts loaded from {CSV_PATH}\n")


os.makedirs(SAVE_DIR, exist_ok=True)
all_results = []

out_csv = os.path.join(SAVE_DIR, "results.csv")
out_fields = ["run", "prompt", "time_bl", "time_dc", "time_adc",
              "speedup_dc", "speedup_adc", "psnr_dc", "psnr_adc",
              "img_sim_dc", "img_sim_adc", "clip_bl", "clip_dc", "clip_adc",
              "dc_full", "dc_cache", "adc_full", "adc_cache",
              "budget_K", "budget_match",
              "adc_intervals", "adc_trace"]

with open(out_csv, "w", newline="", encoding="utf-8") as cf:
    cw = csv.DictWriter(cf, fieldnames=out_fields)
    cw.writeheader()

    for i, prompt in enumerate(prompts):
        print(f"\n{'='*60}\n  Run {i+1}/{len(prompts)}  [Budget K={FULL_BUDGET}]\n{'='*60}")
        print(f'  Prompt: "{prompt[:80]}{"..." if len(prompt)>80 else ""}"')

        # Step 1: Baseline (full DDIM, no caching)
        print("  [1/3] Baseline ...")
        img_bl, t_bl = gen_baseline(prompt, NUM_STEPS, SEED)
        print(f"        {t_bl:.2f}s")

        # Step 2: Official DeepCache (uniform caching)
        print(f"  [2/3] DeepCache (interval={DC_INTERVAL}, branch={DC_BRANCH}) ...")
        img_dc, t_dc = gen_deepcache(prompt, NUM_STEPS, SEED, DC_INTERVAL, DC_BRANCH)
        sp_dc = t_bl / max(t_dc, 1e-6)
        print(f"        {t_dc:.2f}s  ({sp_dc:.2f}x)")

        # Step 3: Adaptive interval with budget constraint
        print(f"  [3/3] Adaptive Interval (budget K={FULL_BUDGET}) ...")
        img_ad, t_ad, stats = gen_adaptive(prompt, NUM_STEPS, SEED, adaptive_config)
        sp_ad = t_bl / max(t_ad, 1e-6)
        ft = ''.join(stats.get('full_trace', []))
        adc_full = stats.get('full_steps', 0)
        adc_cache = stats.get('cache_steps', 0)
        budget_match = (adc_full == FULL_BUDGET)

        print(f"        {t_ad:.2f}s  ({sp_ad:.2f}x)")
        print(f"        Full={adc_full} Cache={adc_cache}  Budget match={'MATCH' if budget_match else 'MISMATCH!'}")
        print(f"        Trace: {ft}")
        print(f"        Intervals: {stats.get('interval_trace', [])}")
        print(f"        Deltas: {[f'{d:.4f}' for d in stats.get('delta_trace', [])]}")

        if not budget_match:
            print(f"  WARNING: Adaptive used {adc_full} Full but budget is {FULL_BUDGET}!")

        # DeepCache full/cache step counts (for verification)
        dc_full = FULL_BUDGET
        dc_cache = NUM_STEPS - dc_full

        # Compute all quality metrics
        psnr_dc = compute_psnr(img_bl, img_dc)
        psnr_ad = compute_psnr(img_bl, img_ad)
        sim_dc = compute_clip_image_similarity(img_bl, img_dc)
        sim_ad = compute_clip_image_similarity(img_bl, img_ad)
        clip_bl = compute_clip_score(img_bl, prompt)
        clip_dc = compute_clip_score(img_dc, prompt)
        clip_ad = compute_clip_score(img_ad, prompt)

        print(f"  PSNR    DC={psnr_dc:.1f}dB  ADC={psnr_ad:.1f}dB")
        print(f"  ImgSim  DC={sim_dc:.4f}  ADC={sim_ad:.4f}")
        print(f"  CLIP    BL={clip_bl:.4f}  DC={clip_dc:.4f}  ADC={clip_ad:.4f}")

        # Save side-by-side comparison image
        W, H = img_bl.size
        comp = Image.new("RGB", (W * 3 + 20, H + 30), "white")
        draw = ImageDraw.Draw(comp)
        for j, (img, title) in enumerate(zip(
            [img_bl, img_dc, img_ad],
            ["Baseline",
             f"DeepCache(i={DC_INTERVAL},F={dc_full})",
             f"Adaptive(F={adc_full})"]
        )):
            x = j * (W + 10)
            comp.paste(img, (x, 25))
            draw.text((x + W // 2, 5), title, fill="black", anchor="mt")
        comp_path = os.path.join(SAVE_DIR, f"cmp_{i+1}_s{SEED}.png")
        comp.save(comp_path)
        print(f"  Saved -> {comp_path}")
        try:
            from IPython.display import display
            display(comp)
        except:
            pass

        # Record results
        r = {
            'prompt': prompt,
            'times': {'baseline': t_bl, 'deepcache': t_dc, 'adaptive': t_ad},
            'speedups': {'deepcache': sp_dc, 'adaptive': sp_ad},
            'psnr': {'deepcache': psnr_dc, 'adaptive': psnr_ad},
            'img_sim': {'deepcache': sim_dc, 'adaptive': sim_ad},
            'clip': {'baseline': clip_bl, 'deepcache': clip_dc, 'adaptive': clip_ad},
            'stats': stats,
            'budget_match': budget_match,
        }
        all_results.append(r)

        # Write CSV row
        cw.writerow({
            "run": i + 1, "prompt": prompt[:100],
            "time_bl": f"{t_bl:.3f}", "time_dc": f"{t_dc:.3f}", "time_adc": f"{t_ad:.3f}",
            "speedup_dc": f"{sp_dc:.3f}", "speedup_adc": f"{sp_ad:.3f}",
            "psnr_dc": f"{psnr_dc:.2f}", "psnr_adc": f"{psnr_ad:.2f}",
            "img_sim_dc": f"{sim_dc:.4f}", "img_sim_adc": f"{sim_ad:.4f}",
            "clip_bl": f"{clip_bl:.4f}", "clip_dc": f"{clip_dc:.4f}", "clip_adc": f"{clip_ad:.4f}",
            "dc_full": dc_full, "dc_cache": dc_cache,
            "adc_full": adc_full, "adc_cache": adc_cache,
            "budget_K": FULL_BUDGET,
            "budget_match": "YES" if budget_match else "NO",
            "adc_intervals": str(stats.get('interval_trace', [])),
            "adc_trace": ft,
        })
        cf.flush()

# ============================================================
# 7. Summary
# ============================================================

print(f"\n{'='*70}")
print(f"  SUMMARY ({len(all_results)} runs, Budget K={FULL_BUDGET})")
print(f"{'='*70}")
sp_dc = np.mean([r['speedups']['deepcache'] for r in all_results])
sp_ad = np.mean([r['speedups']['adaptive'] for r in all_results])
p_dc = np.mean([r['psnr']['deepcache'] for r in all_results])
p_ad = np.mean([r['psnr']['adaptive'] for r in all_results])
s_dc = np.mean([r['img_sim']['deepcache'] for r in all_results])
s_ad = np.mean([r['img_sim']['adaptive'] for r in all_results])
c_bl = np.mean([r['clip']['baseline'] for r in all_results])
c_dc = np.mean([r['clip']['deepcache'] for r in all_results])
c_ad = np.mean([r['clip']['adaptive'] for r in all_results])
budget_ok = sum(1 for r in all_results if r['budget_match'])

print(f"  Budget match: {budget_ok}/{len(all_results)} runs")
print()
print(f"  {'Method':<28} {'Speedup':<10} {'PSNR(dB)':<12} {'ImgSim':<10} {'CLIP':<10} {'Full Steps'}")
print(f"  {'-'*82}")
print(f"  {'DDIM Baseline':<28} {'1.00x':<10} {'--':<12} {'--':<10} {c_bl:.4f}{'':>4} {NUM_STEPS}")
print(f"  {'DeepCache(i='+str(DC_INTERVAL)+')':<28} {sp_dc:.2f}x{'':>4} {p_dc:.1f}{'':>7} {s_dc:.4f}{'':>3} {c_dc:.4f}{'':>3} {FULL_BUDGET}")
print(f"  {'Adaptive(budget='+str(FULL_BUDGET)+')':<28} {sp_ad:.2f}x{'':>4} {p_ad:.1f}{'':>7} {s_ad:.4f}{'':>3} {c_ad:.4f}{'':>3} {FULL_BUDGET}")
print()
print(f"  Both methods use exactly {FULL_BUDGET} full UNet evaluations.")
print(f"  Only difference: DeepCache=uniform (every {DC_INTERVAL} steps), Adaptive=δ-driven (front-heavy).")
print(f"{'='*70}")
print(f"  CSV -> {out_csv}")
print("Done!")